# 06 - Carte interactive multi-thématique

Carte Folium multi-couches au niveau départemental couvrant les 3 domaines d'analyse :
- **Logement** : taux logement social (SRU), loyer moyen au m²
- **Éducation** : ratio élèves/enseignant, % écoles en éducation prioritaire
- **Revenus** : niveau de vie médian, taux de pauvreté

In [1]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import folium

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import DATA_PROCESSED, DATA_EXTERNAL, OUTPUTS

## 6.1 Chargement des données

In [2]:
sru_dep = pd.read_parquet(DATA_PROCESSED / "sru_par_departement.parquet")
loyer_dep = pd.read_parquet(DATA_PROCESSED / "loyers_par_departement.parquet")
edu_dep = pd.read_parquet(DATA_PROCESSED / "education_par_departement.parquet")

filo_dep_path = DATA_PROCESSED / "filosofi_par_departement.parquet"
if filo_dep_path.exists():
    filo_dep = pd.read_parquet(filo_dep_path)
    print(f"FiloSoFi/d\u00e9p: {filo_dep.shape}")
else:
    filo_dep = pd.DataFrame()
    print("FiloSoFi/d\u00e9p: non disponible")

print(f"SRU/d\u00e9p: {sru_dep.shape}")
print(f"Loyers/d\u00e9p: {loyer_dep.shape}")
print(f"\u00c9ducation/d\u00e9p: {edu_dep.shape}")

FiloSoFi/dép: non disponible
SRU/dép: (94, 7)
Loyers/dép: (100, 3)
Éducation/dép: (101, 10)


In [3]:
with open(DATA_EXTERNAL / "departements.geojson", encoding="utf-8") as f:
    geojson_data = json.load(f)

print(f"GeoJSON: {len(geojson_data['features'])} départements")
print(f"Propriétés: {list(geojson_data['features'][0]['properties'].keys())}")

GeoJSON: 96 départements
Propriétés: ['code', 'nom']


## 6.2 Construction de la carte

In [4]:
geo_code_field = "code"

m = folium.Map(
    location=[46.603354, 1.888334],
    zoom_start=6,
    tiles="cartodbpositron",
    width="100%",
    height="80%",
)

In [5]:
def add_choropleth(map_obj, geojson, data, code_field, value_field, legend_name, colors, layer_name):
    g = folium.Choropleth(
        geo_data=geojson,
        data=data,
        columns=[code_field, value_field],
        key_on=f"feature.properties.{code_field}",
        fill_color=colors,
        fill_opacity=0.7,
        line_opacity=0.3,
        legend_name=legend_name,
        name=layer_name,
        highlight=True,
    )
    g.add_to(map_obj)
    return g

In [6]:
if not filo_dep.empty:
    g1 = add_choropleth(
        m, geojson_data, sru_dep,
        "code_departement", "taux_sru_moyen",
        "Taux moyen logements sociaux (%)",
        "YlGn", "Logement social (SRU)",
    )

    g2 = add_choropleth(
        m, geojson_data, loyer_dep,
        "code_departement", "loyer_moyen_m2",
        "Loyer moyen au m² (€)",
        "YlOrRd", "Loyer moyen",
    )

    g3 = add_choropleth(
        m, geojson_data, edu_dep,
        "code_departement", "ratio_eleves_enseignant",
        "Ratio élèves / enseignant",
        "YlOrRd", "Ratio élèves/enseignant",
    )

    g4 = add_choropleth(
        m, geojson_data, edu_dep,
        "code_departement", "pct_ecoles_rep",
        "% écoles REP/REP+",
        "OrRd", "Éducation prioritaire",
    )

    pauvrete_col = [c for c in filo_dep.columns if "pauvreté" in c.lower()][0]
    g5 = add_choropleth(
        m, geojson_data, filo_dep,
        "code_departement", pauvrete_col,
        "Taux de pauvreté (%)",
        "Reds", "Taux de pauvreté",
    )

    mediane_col = [c for c in filo_dep.columns if "médiane" in c.lower()][0]
    g6 = add_choropleth(
        m, geojson_data, filo_dep,
        "code_departement", mediane_col,
        "Niveau de vie médian (€/an)",
        "YlGn", "Niveau de vie médian",
    )

    print("6 couches ajoutées")


## 6.3 Tooltips

In [7]:
if not filo_dep.empty:
    tooltip_data = sru_dep[["code_departement", "taux_sru_moyen"]].merge(
        loyer_dep[["code_departement", "loyer_moyen_m2"]], on="code_departement", how="outer"
    ).merge(
        edu_dep[["code_departement", "ratio_eleves_enseignant", "pct_ecoles_rep"]], on="code_departement", how="outer"
    ).merge(
        filo_dep[["code_departement", pauvrete_col, mediane_col]], on="code_departement", how="outer"
    )

    tooltip_lookup = {}
    for _, row in tooltip_data.iterrows():
        code = row["code_departement"]
        tooltip_lookup[code] = {
            "Logement social": f"{row['taux_sru_moyen']:.1f}%" if pd.notna(row["taux_sru_moyen"]) else "N/A",
            "Loyer moyen": f"{row['loyer_moyen_m2']:.1f} €/m²" if pd.notna(row["loyer_moyen_m2"]) else "N/A",
            "Ratio él/ens": f"{row['ratio_eleves_enseignant']:.1f}" if pd.notna(row["ratio_eleves_enseignant"]) else "N/A",
            "% EP": f"{row['pct_ecoles_rep']:.1f}%" if pd.notna(row["pct_ecoles_rep"]) else "N/A",
            "Pauvreté": f"{row[pauvrete_col]:.1f}%" if pd.notna(row[pauvrete_col]) else "N/A",
            "Niveau de vie": f"{row[mediane_col]:,.0f} €" if pd.notna(row[mediane_col]) else "N/A",
        }
else:
    tooltip_data = sru_dep[["code_departement", "taux_sru_moyen"]].merge(
        loyer_dep[["code_departement", "loyer_moyen_m2"]], on="code_departement", how="outer"
    ).merge(
        edu_dep[["code_departement", "ratio_eleves_enseignant", "pct_ecoles_rep"]], on="code_departement", how="outer"
    )

    tooltip_lookup = {}
    for _, row in tooltip_data.iterrows():
        code = row["code_departement"]
        tooltip_lookup[code] = {
            "Logement social": f"{row['taux_sru_moyen']:.1f}%" if pd.notna(row["taux_sru_moyen"]) else "N/A",
            "Loyer moyen": f"{row['loyer_moyen_m2']:.1f} €/m²" if pd.notna(row["loyer_moyen_m2"]) else "N/A",
            "Ratio él/ens": f"{row['ratio_eleves_enseignant']:.1f}" if pd.notna(row["ratio_eleves_enseignant"]) else "N/A",
            "% EP": f"{row['pct_ecoles_rep']:.1f}%" if pd.notna(row["pct_ecoles_rep"]) else "N/A",
        }


In [8]:
tooltip = folium.GeoJsonTooltip(
    fields=["nom", "code"],
    aliases=["Département", "Code"],
    localize=True,
    sticky=True,
)

for feature in geojson_data["features"]:
    code = feature["properties"]["code"]
    if code in tooltip_lookup:
        info = tooltip_lookup[code]
        extra = "<br>".join([f"<b>{k}</b>: {v}" for k, v in info.items()])
        feature["properties"]["_tooltip_extra"] = extra

tooltip_full = folium.GeoJsonTooltip(
    fields=["nom", "code", "_tooltip_extra"],
    aliases=["Département", "Code", ""],
    localize=True,
    sticky=True,
    labels=False,
)

folium.GeoJson(
    geojson_data,
    tooltip=tooltip_full,
    style_function=lambda x: {"fillOpacity": 0, "color": "gray", "weight": 0.5},
    highlight_function=lambda x: {"weight": 2, "color": "black"},
).add_to(m)

print("Tooltips ajoutés")

Tooltips ajoutés


## 6.4 Contrôles et titre

In [9]:
folium.LayerControl(collapsed=True).add_to(m)

title_html = """
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
     background-color: white; padding: 10px 20px; border-radius: 5px;
     box-shadow: 0 2px 6px rgba(0,0,0,0.3); font-family: sans-serif;">
  <h3 style="margin: 0; color: #2c3e50;">Enjeux locaux des communes - Élections municipales 2026</h3>
  <p style="margin: 5px 0 0 0; font-size: 12px; color: #7f8c8d;">Logement | Éducation | Revenus & Pauvreté</p>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

print("Contrôles et titre ajoutés")

Contrôles et titre ajoutés


## 6.5 Export

In [10]:
output_path = OUTPUTS / "carte_interactive.html"
m.save(str(output_path))
print(f"Carte exportée: {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)")

Carte exportée: C:\Users\Salty\Downloads\defi-municipales-2026-enjeux\outputs\carte_interactive.html (0.6 MB)
